In [0]:
# Feature-drift gate. Two independent signals (either one sets drift_detected):
#   1. Lakehouse Monitoring drift-metrics table: KS test per feature, latest window (as in the fraud reference)
#   2. PSI (+ local KS statistic) of the latest inference window vs. the training-period feature distribution
# Drift triggers investigation/retraining only; a retrained model must still pass evaluate.py before promotion.
import sys
import os
sys.path.insert(0, os.getcwd())

import numpy as np
from pyspark.sql import functions as F
from utils import (as_bool, chronological_cuts, ks_statistic, load_project, population_stability_index,
                   psi_level, table, widget_or)

cfg = load_project()
mon = cfg["monitoring"]

dbutils.widgets.text("catalog", "stock_demo")
dbutils.widgets.text("psi_moderate", "")
dbutils.widgets.text("psi_high", "")
dbutils.widgets.text("min_rows_per_window", "")
dbutils.widgets.text("retrain_on_drift", "")
dbutils.widgets.text("source_log_table", "")    # empty -> inference log; set to test the drift path on another table

catalog = dbutils.widgets.get("catalog")
moderate = float(widget_or(dbutils.widgets.get("psi_moderate"), mon["psi_moderate"]))
high = float(widget_or(dbutils.widgets.get("psi_high"), mon["psi_high"]))
min_rows = int(widget_or(dbutils.widgets.get("min_rows_per_window"), mon["min_rows_per_window"]))
retrain_on_drift = as_bool(widget_or(dbutils.widgets.get("retrain_on_drift"), mon["retrain_on_drift"]))
FEATURES = list(mon["drift_features"])

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.monitoring")
log_table = widget_or(dbutils.widgets.get("source_log_table"), table(cfg, "inference_log", catalog))

# ---------------------------------------------------------------------------------
# Signal 1: Lakehouse Monitoring KS test (skipped when the monitor has not produced metrics yet)
# ---------------------------------------------------------------------------------
ks_breached = 0
drift_metrics_table = f"{catalog}.monitoring.{log_table.split('.')[-1]}_drift_metrics"
if spark.catalog.tableExists(drift_metrics_table):
    dm = spark.table(drift_metrics_table)
    latest_window = dm.select(F.max(F.col("window.start")).alias("w")).collect()[0]["w"]
    if latest_window is not None:
        ks_breached = (dm.filter(F.col("window.start") == latest_window)
                         .filter(F.col("column_name").isin(FEATURES))
                         .filter((F.col("ks_test.statistic") > float(mon["ks_statistic_threshold"])) &
                                 (F.col("ks_test.pvalue") < float(mon["ks_pvalue_threshold"])))
                         .count())
    print(f"Monitor KS breaches in latest window: {ks_breached}")
else:
    print(f"{drift_metrics_table} not found - run the monitor refresh first. Skipping signal 1.")

# ---------------------------------------------------------------------------------
# Signal 2: PSI / KS of the latest inference window vs. the training-period reference
# ---------------------------------------------------------------------------------
rows, psi_high_count, window_label, status = [], 0, None, "skipped"
if spark.catalog.tableExists(log_table):
    log = spark.table(log_table).withColumn("_day", F.to_date("request_ts"))
    latest_day = log.select(F.max("_day")).collect()[0][0]
    current = log.filter(F.col("_day") == latest_day).select(*FEATURES).toPandas() if latest_day else None

    # The reference only needs a representative sample, not millions of rows on the driver.
    feat_sdf = spark.table(table(cfg, "features", catalog)).select("trade_ts", *FEATURES)
    MAX_REFERENCE_ROWS = 500_000
    total_rows = feat_sdf.count()
    if total_rows > MAX_REFERENCE_ROWS:
        feat_sdf = feat_sdf.sample(fraction=MAX_REFERENCE_ROWS / total_rows, seed=42)
    feats = feat_sdf.toPandas()
    feats = feats.replace([np.inf, -np.inf], np.nan).dropna().sort_values("trade_ts")
    cut1, _ = chronological_cuts(feats.trade_ts.unique(), cfg["model"]["validation_fraction"],
                                 cfg["model"]["test_fraction"])
    reference = feats[feats.trade_ts <= cut1]

    if current is not None:
        current = current.replace([np.inf, -np.inf], np.nan).dropna()
    if current is None or len(current) < min_rows or len(reference) < min_rows:
        print(f"Not enough rows for PSI (current={0 if current is None else len(current)}, "
              f"reference={len(reference)}, min={min_rows}). Skipping signal 2.")
    else:
        window_label, status = str(latest_day), "computed"
        for col in FEATURES:
            psi = population_stability_index(reference[col].values, current[col].values)
            ks = ks_statistic(reference[col].values, current[col].values)
            rows.append((col, float(psi), float(ks), psi_level(psi, moderate, high)))
        psi_high_count = sum(1 for r in rows if r[3] == "HIGH")
        for r in rows:
            print(f"{r[0]:>15}: PSI={r[1]:.3f} KS={r[2]:.3f} -> {r[3]}")
else:
    print(f"{log_table} not found. Skipping signal 2.")

if rows:
    (spark.createDataFrame(rows, ["feature_name", "psi", "ks_statistic", "drift_level"])
          .withColumn("window_day", F.lit(window_label))
          .withColumn("source_table", F.lit(log_table))
          .withColumn("checked_at", F.current_timestamp())
          .write.format("delta").mode("append").option("mergeSchema", "true")
          .saveAsTable(table(cfg, "drift", catalog)))

detected = (ks_breached > 0) or (psi_high_count > 0)
print(f"Drift detected: {detected} (monitor KS breaches={ks_breached}, PSI HIGH features={psi_high_count}, "
      f"PSI status={status}); retrain_on_drift={retrain_on_drift}")

dbutils.jobs.taskValues.set(key="drift_detected", value=bool(detected))
dbutils.jobs.taskValues.set(key="retrain_required", value=bool(detected and retrain_on_drift))